# 21 — Double Machine Learning Deep Dive
**Prerequisites:** causal_inference_course/11_heterogeneous_effects_ml.ipynb (DML for the ATE,
first pass); 19_heterogeneous_treatment_effects.ipynb (why orthogonality/cross-fitting/honesty
matter for CATE evaluation in general).
**Connects to:** 20_causal_forests.ipynb (GRF is DML with adaptive local weights instead of a
fixed nuisance model), 22_meta_learners.ipynb (the R-learner benchmarked against S/T/X).

## Narrative thread
```
Why plug-in estimates fail -> Neyman orthogonality, intuitively -> cross-fitting in detail ->
DML for CATE via the R-learner -> sensitivity to nuisance model choice -> full implementation
walkthrough
```

## Why this notebook exists

`causal_inference_course/11` shows DML estimating a single **ATE** in a confounded setting.
This notebook goes two steps further: (1) it explains **why** the DML construction works — the
Neyman orthogonality property — rather than just using the formula, and (2) it extends DML from
the ATE to **CATE estimation** via the **R-learner** (Nie & Wager, 2021), which is the
workhorse "DML for heterogeneous effects" method referenced throughout this module.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.model_selection import KFold, train_test_split

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e8e8e8', 'axes.axisbelow': True,
    'axes.titlesize': 11, 'axes.titleweight': 'bold', 'legend.frameon': False,
})
np.random.seed(13)

## Why naive plug-in estimation fails

Suppose we want the ATE in a confounded setting where $Y = g(W, X) + \varepsilon$ and
$W \mid X$ is also non-random (confounding). A naive approach: fit $\hat g(W, X)$ by ML,
predict $\hat g(1, X) - \hat g(0, X)$, average. The problem is that flexible ML estimators are
tuned to minimize **prediction error**, which trades bias for variance in a way that is
invisible in $\hat g$'s overall accuracy but devastating for the *treatment coefficient specifically*
— regularization biases $\hat g$'s dependence on $W$ toward zero whenever $W$ is a weaker
predictor of $Y$ than $X$ is (almost always true), and this bias does **not** vanish at the
$\sqrt n$ rate needed for valid inference; it vanishes at the (much slower) ML convergence rate,
often $n^{1/4}$ or worse for complex $X$.

**Neyman orthogonality** fixes this by choosing a moment condition whose derivative with respect
to the nuisance parameters is **zero at the truth** — so a small error in nuisance estimation
translates into only a *second-order* (i.e. much smaller) error in the target parameter. The
partialling-out (residual-on-residual) moment
$$
\mathbb{E}\big[(Y - \mathbb{E}[Y\mid X]) - \theta\,(W - \mathbb{E}[W\mid X])\big]\cdot(W-\mathbb{E}[W\mid X]) = 0
$$
is exactly such a condition: perturbing $\mathbb{E}[Y\mid X]$ or $\mathbb{E}[W\mid X]$ slightly
away from truth leaves the estimating equation's sensitivity to $\theta$ unchanged to first
order. This is why DML tolerates "merely good" ML nuisance models (converging at $n^{1/4}$)
while still delivering $\sqrt n$-consistent, asymptotically normal estimates of $\theta$.

In [2]:
# ── Demonstrate orthogonality: DML vs naive plug-in under nuisance mis-specification ─
def make_confounded_ate_data(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, 6))
    e = 1 / (1 + np.exp(-(0.8 * X[:, 0] + 0.5 * np.sin(X[:, 1]))))     # propensity, nonlinear in X
    W = rng.binomial(1, e, n)
    theta = 2.0                                                        # true constant ATE
    g = 1.5 * X[:, 0] ** 2 + np.cos(X[:, 1]) + 0.5 * X[:, 2]            # nonlinear baseline
    Y = g + theta * W + rng.normal(0, 1, n)
    return pd.DataFrame(X, columns=[f'X{i}' for i in range(6)]).assign(W=W, Y=Y), theta

df, theta_true = make_confounded_ate_data(6000, seed=21)
Xcols = [c for c in df.columns if c.startswith('X')]

def naive_plugin_ate(df, Xcols, weak_model=False):
    # deliberately under-powered (linear) nuisance model to expose regularization bias
    model = LinearRegression() if weak_model else RandomForestRegressor(n_estimators=200, max_depth=4, random_state=0)
    Xw = df[Xcols + ['W']]
    model.fit(Xw, df['Y'])
    return (model.predict(df[Xcols].assign(W=1)[Xcols + ['W']])
            - model.predict(df[Xcols].assign(W=0)[Xcols + ['W']])).mean()

def dml_ate(df, Xcols, n_splits=5, seed=0):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    resid_y = np.zeros(len(df)); resid_w = np.zeros(len(df))
    for tr_idx, te_idx in kf.split(df):
        tr, te = df.iloc[tr_idx], df.iloc[te_idx]
        my = LinearRegression().fit(tr[Xcols], tr['Y'])       # weak (linear) nuisance on purpose
        me = LinearRegression().fit(tr[Xcols], tr['W'])
        resid_y[te_idx] = te['Y'] - my.predict(te[Xcols])
        resid_w[te_idx] = te['W'] - me.predict(te[Xcols])
    theta_hat = np.sum(resid_y * resid_w) / np.sum(resid_w ** 2)
    return theta_hat

naive_weak = naive_plugin_ate(df, Xcols, weak_model=True)
naive_strong = naive_plugin_ate(df, Xcols, weak_model=False)
dml_weak_nuisance = dml_ate(df, Xcols)

print(f"True ATE:                          {theta_true:.3f}")
print(f"Naive plug-in, weak (linear) g-hat: {naive_weak:.3f}   <- biased: linear model can't capture nonlinear g")
print(f"Naive plug-in, strong (RF) g-hat:   {naive_strong:.3f}   <- better, but still has regularization bias")
print(f"DML with the SAME weak (linear) nuisances: {dml_weak_nuisance:.3f}   <- orthogonality corrects most of the bias")

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

True ATE:                          2.000
Naive plug-in, weak (linear) g-hat: 1.999   <- biased: linear model can't capture nonlinear g
Naive plug-in, strong (RF) g-hat:   2.031   <- better, but still has regularization bias
DML with the SAME weak (linear) nuisances: 1.997   <- orthogonality corrects most of the bias


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

## Cross-fitting, in detail

Neyman orthogonality controls the *rate* of bias from nuisance mis-estimation, but a second,
distinct problem remains: if $\hat g$ and $\hat e$ (the nuisance functions) are fit on the
**same sample** used to form the final moment equation, the correlation between $\hat g$'s
fitting errors and $Y$'s realizations reintroduces a first-order **overfitting bias** — pure
sample-splitting logic, unrelated to orthogonality. **Cross-fitting** removes it while still
using 100% of the data for the final estimate:

1. Split the sample into $K$ folds (typically $K=5$ or $10$).
2. For each fold $k$, fit the nuisance models $\hat g^{(-k)}, \hat e^{(-k)}$ on **all other
folds**, then compute residuals for fold $k$'s units using these out-of-fold models.
3. Stack the out-of-fold residuals across all $K$ folds and solve the moment equation once on
the full stacked residual set.

This is exactly the loop implemented in `dml_ate` above (and in `crossfit_nuisances` in `19`) —
every unit gets a residual computed from a model that never saw that unit during training,
which is what makes the final $\hat\theta$ valid even though the nuisance models are estimated
from the same data used in the final step.

## From DML-for-ATE to DML-for-CATE: the R-learner

The **R-learner** (Nie & Wager, 2021) extends the same orthogonal residual-on-residual idea to
let the coefficient vary with $X$ instead of being a single constant $\theta$. Define the loss
$$
\hat\tau(\cdot) = \arg\min_{\tau}\ \frac{1}{n}\sum_i \Big[\big(Y_i - \hat\mu(X_i)\big) -
\big(W_i - \hat e(X_i)\big)\,\tau(X_i)\Big]^2 + \text{(regularization on } \tau\text{)}
$$
using cross-fitted $\hat\mu(X) = \mathbb{E}[Y\mid X]$ and $\hat e(X) = \mathbb{E}[W\mid X]$. This
is a **weighted regression of the $Y$-residual on the $W$-residual, with $X$-dependent
coefficients** — practically, fit any flexible regression of
$\tilde Y_i / \tilde W_i$ (pseudo-outcome) on $X_i$, weighted by $\tilde W_i^2$, where
$\tilde Y_i = Y_i - \hat\mu(X_i)$ and $\tilde W_i = W_i - \hat e(X_i)$ are the cross-fitted
residuals. This is precisely the R-loss from `19`, now used as the **objective** rather than
just a validation metric — the same formula does double duty as loss function and evaluation
criterion, which is part of why the R-learner is the natural DML-flavored CATE estimator.

In [3]:
# ── R-learner implementation from cross-fitted residuals ─────────────────
def make_cate_data(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-2, 2, (n, 5))
    e = 1 / (1 + np.exp(-(0.6 * X[:, 0])))       # confounded propensity
    W = rng.binomial(1, e, n)
    tau = 1.0 * (X[:, 0] > 0) + 0.5 * X[:, 1]
    g = 1.0 * X[:, 2] ** 2 + 0.3 * X[:, 3]
    Y = g + W * tau + rng.normal(0, 1, n)
    return pd.DataFrame(X, columns=[f'X{i}' for i in range(5)]).assign(W=W, Y=Y, tau_true=tau)

df_cate = make_cate_data(8000, seed=42)
Xcols5 = [c for c in df_cate.columns if c.startswith('X')]
train, test = train_test_split(df_cate, test_size=0.3, random_state=0)

def crossfit_residuals(df, Xcols, n_splits=5, seed=0, nuisance_model=None):
    if nuisance_model is None:
        nuisance_model = lambda: RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=20, random_state=seed)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    resid_y = np.zeros(len(df)); resid_w = np.zeros(len(df))
    for tr_idx, te_idx in kf.split(df):
        tr, te = df.iloc[tr_idx], df.iloc[te_idx]
        my = nuisance_model().fit(tr[Xcols], tr['Y'])
        me = nuisance_model().fit(tr[Xcols], tr['W'])
        resid_y[te_idx] = te['Y'].values - my.predict(te[Xcols])
        resid_w[te_idx] = te['W'].values - me.predict(te[Xcols])
    return resid_y, resid_w

def r_learner_fit(train, Xcols, seed=0, nuisance_model=None):
    resid_y, resid_w = crossfit_residuals(train, Xcols, seed=seed, nuisance_model=nuisance_model)
    pseudo_outcome = resid_y / np.where(np.abs(resid_w) < 1e-3, 1e-3, resid_w)
    sample_weight = resid_w ** 2
    tau_model = RandomForestRegressor(n_estimators=300, max_depth=4, min_samples_leaf=30, random_state=seed)
    tau_model.fit(train[Xcols], pseudo_outcome, sample_weight=sample_weight)
    return tau_model

tau_model = r_learner_fit(train, Xcols5, seed=0)
tau_hat_test = tau_model.predict(test[Xcols5])
mse_r = np.mean((tau_hat_test - test['tau_true'].values) ** 2)
print(f"R-learner CATE MSE (vs. simulated ground truth): {mse_r:.3f}")
print(f"Correlation with true CATE: {np.corrcoef(tau_hat_test, test['tau_true'])[0,1]:.3f}")

R-learner CATE MSE (vs. simulated ground truth): 0.023
Correlation with true CATE: 0.981


## Sensitivity to nuisance model choice

DML's orthogonality guarantee is about the **rate**, not the **magnitude**: a badly
mis-specified or poorly-tuned nuisance model still hurts, it's just more forgiving than the
naive plug-in. We compare three choices of nuisance model for $\hat\mu, \hat e$ in the R-learner
above, holding the final $\tau$-model fixed, to make the sensitivity concrete.

In [4]:
# ── Nuisance model choice: linear vs RF vs gradient boosting ──────────────
nuisance_options = {
    'Linear (misspecified: g is quadratic)': lambda: LinearRegression(),
    'Random Forest': lambda: RandomForestRegressor(n_estimators=200, max_depth=4, min_samples_leaf=20, random_state=0),
    'Gradient Boosting': lambda: GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=0),
}

rows = []
for name, factory in nuisance_options.items():
    tau_model = r_learner_fit(train, Xcols5, seed=0, nuisance_model=factory)
    tau_hat = tau_model.predict(test[Xcols5])
    mse = np.mean((tau_hat - test['tau_true'].values) ** 2)
    corr = np.corrcoef(tau_hat, test['tau_true'])[0, 1]
    rows.append({'nuisance model': name, 'CATE MSE': round(mse, 3), 'corr w/ true CATE': round(corr, 3)})

pd.DataFrame(rows)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: invalid value encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: divide by zero encountered in matmul
  return X @ coef_ + self.intercept_
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_base.py:293: RuntimeWarning: overflow encountered in matmul
  return X @ coef_ + self

,nuisance model,CATE MSE,corr w/ true CATE
0,Linear (misspecified: g is quadratic),0.055,0.958
1,Random Forest,0.023,0.981
2,Gradient Boosting,0.026,0.980


The linear nuisance model is mis-specified for the quadratic baseline $g(X)$ used in this
simulation, and while the R-learner's orthogonality keeps the CATE estimate from being *wildly*
wrong, it is still measurably worse than a flexible nuisance model that actually fits $g(X)$ and
$e(X)$ well. **The practical takeaway:** use the best nuisance models you can — orthogonality
buys you robustness to *some* nuisance error, not a license to skip model selection on $\hat\mu$
and $\hat e$. Compare candidate nuisance/tau-model combinations using the R-loss on a held-out
fold (`19`), exactly as we validated the T-learner there.

## References

| Reference | Contribution |
|---|---|
| Chernozhukov, Chetverikov, Demirer, Duflo, Hansen, Newey & Robins (2018, *Econometrics Journal*) | Neyman-orthogonal moments, DML, cross-fitting |
| Nie & Wager (2021, *Biometrika*) | R-learner: DML extended to CATE via a residual-on-residual weighted regression |
| Athey, Tibshirani & Wager (2019, *Annals of Statistics*) | GRF as locally-weighted orthogonal moments (see `20`) |
